# TSFS12 Hand-in exercise 4: Collaborative Control 6.2

In [285]:
import numpy as np
from scipy.integrate import odeint
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.animation as animation
from collab_functions import CreateAgent, AgentFormation

It is necessary to plot in an external windows for the animations to work.
Run the ipython-command below to activate plotting in external windows.

If you have no default Matplotlib backend, you might have to write

```%matplotlib tk``` 

to explicitly state which backend to use (tk is a good choice that should work on all platforms)

In [286]:
%matplotlib qt

# Model, controllers, and agent formation
A first example are 4 agents, all modeled by a single integrator, and all measuring their absolute position.  The controller for each agent is a proportional controller.

In [287]:
def single_integrator(t, x, u, model_params):
    return u


def controller_relative(y, y_ref, ctrl_params):
    u = np.zeros(2)
    for relative_position, desired_distance in zip(y, y_ref):
        distance_squared = relative_position @ relative_position
        u += 4 * ctrl_params["kp"] * (distance_squared - desired_distance**2) * relative_position
    return u


Now, formulate the formation graph and position references. The graph is specified by a set of edges. An edge (i, j) represents that agent j meauseras the distance from agent i. If agent i measures its own absolute position, this is represented by a tuple with element i as (i,) and this is the case for all agents in this example.

In [288]:
n = 2  # (px, py)

GRAPH = "right"
INITIAL_CONDITION = 1

# Agent order: blue, orange, green, red.
# With the right graph: case 1 gives a square; case 2 makes blue overlap red.
# All agents start away from their reference corners. Red has no measurements
# and stays fixed; the other three move to form the shape around it.
initial_position_sets = {
    1: np.array([
        [-0.7, -0.6],
        [1.8, -0.6],
        [-0.6, 1.7],
        [1.2, 1.3],
    ]),
    2: np.array([
        [1.7, 1.8],
        [1.8, -0.6],
        [-0.6, 1.7],
        [1.2, 1.3],
    ]),
}
initial_positions = initial_position_sets[INITIAL_CONDITION]

desired_positions = np.array([
    [0.0, 0.0], [1.0, 0.0], [0.0, 1.0], [1.0, 1.0]
])

graphs = {
    "left":  [(3, 2), (3, 1), (1, 0), (0, 2), (1, 2)],
    "right": [(3, 2), (3, 1), (1, 0), (2, 0), (1, 2)],
}
G = graphs[GRAPH]


def distance_reference(agent_idx):
    return np.array([
        np.linalg.norm(desired_positions[i] - desired_positions[j])
        for i, j in G if j == agent_idx
    ])


formation_references = [
    lambda t, i=i: distance_reference(i) for i in range(len(desired_positions))
]


Create all agents by assigning model and controller using ```CreateAgent``` and then form the formation using ```AgentFormation``` by providing the references.

# Simulate and animate
Define an initial state and evaluate the formatation state-transition function

In [289]:
# Controller parameters
ctrl_params = {"kp": 1.0}

# All four agents use the same controller with their available measurements.
agents = [
    CreateAgent(single_integrator, {}, controller_relative, ctrl_params)
    for _ in desired_positions
]
formation = AgentFormation(agents, G, formation_references, n)
x0 = initial_positions.reshape(-1)


Simulate the formation

In [290]:
t = np.arange(0, 1, 0.05) # Time vector for the simulation
x = odeint(formation.ode, x0, t)

Animate the results

In [291]:
fig, (ax, ax_dist) = plt.subplots(
    1, 2, num=20, clear=True, figsize=(12, 6),
    gridspec_kw={"width_ratios": [1.2, 1]}
)
fig.subplots_adjust(left=0.06, right=0.98, wspace=0.28)
positions = x.reshape(-1, len(agents), n)
ax.set_xlim(-1, 2)
ax.set_ylim(-1, 2)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title(f"Distance-based formation control (Kp = {ctrl_params['kp']})")

# Plot each measured edge distance.
for i, j in G:
    distance = np.linalg.norm(positions[:, i] - positions[:, j], axis=1)
    ax_dist.plot(t, distance, label=f"Distance ({i}, {j})")
ax_dist.set_xlim(0, t[-1])
ax_dist.set_ylim(0, 3)
ax_dist.set_xlabel("Time [s]")
ax_dist.set_ylabel("Distance")
ax_dist.set_title("Distances on erroneous edges")
ax_dist.grid(True)
ax_dist.legend()

# One moving marker and one trajectory line per agent
colors = ["tab:blue", "tab:orange", "tab:green", "tab:red", "tab:cyan", "tab:purple", "tab:brown", "tab:pink"]
agents_plot = [ax.plot([], [], "o", color=colors[i])[0] for i in range(len(agents))]
trails = [ax.plot([], [], "-", color=colors[i], alpha=0.7)[0] for i in range(len(agents))]
time_text = ax.text(0.02, 0.95, "", transform=ax.transAxes)
def animate(i):
    for idx, (marker, trail) in enumerate(zip(agents_plot, trails)):
        px = x[: i + 1, n * idx]
        py = x[: i + 1, n * idx + 1]
        marker.set_data([px[-1]], [py[-1]])
        trail.set_data(px, py)
    time_text.set_text(f"t = {t[i]:.1f}")
    return tuple(agents_plot + trails) + (time_text,)

ani = animation.FuncAnimation(
    fig, animate, interval=20, frames=x.shape[0], blit=True, repeat=False
)
xf = x[-1].reshape(len(agents), n)

for i in range(len(agents)):
    print(i, xf[i])

0 [0.55354783 0.02694179]
1 [1.50118776 0.34623877]
2 [0.24325613 0.98525391]
3 [1.2 1.3]
